In [1]:
import pandas as pd
from pathlib import Path

csv_file = Path.cwd() / "analysis_summary.csv"

df = pd.read_csv(csv_file)
df = df.drop(columns=["testmode", "http_passed", "pit_survived"])

df["http_kill_rate"] = df["http_kill_rate"].astype(float)
df["http_kill_rate"] = df["http_kill_rate"].round(2)
df["http_kill_rate"] = df["http_kill_rate"].astype(str) + "%"
df["http_kill_rate"] = df["http_kill_rate"].replace("nan%", "0%")
df["pit_kill_rate"] = df["pit_kill_rate"].astype(float)
df["pit_kill_rate"] = df["pit_kill_rate"].round(2)
df["pit_kill_rate"] = df["pit_kill_rate"].astype(str) + "%"
df["pit_kill_rate"] = df["pit_kill_rate"].replace("nan%", "0%")
# rename columns
# df = df.rename(columns={'pit_survived': 'pit_passed'})
# NaN values to 0
# In level columns, replace reg with regression

df = df.fillna(0)
# 

print(df.columns.tolist())

['sut', 'operationId', 'strength', 'assertlevel', 'http_total', 'http_killed', 'http_kill_rate', 'line_coverage', 'pit_killed', 'pit_total', 'pit_kill_rate']


In [2]:
df

,sut,operationId,strength,assertlevel,http_total,http_killed,http_kill_rate,line_coverage,pit_killed,pit_total,pit_kill_rate
0,youtube,getsearch,two-way,oas,10522,4836,45.96%,76.0,122,511,23.87%
1,youtube,getsearch,two-way,5xx,10522,156,1.48%,76.0,52,511,10.18%
2,youtube,getsearch,two-way,reg,10522,9492,90.21%,76.0,177,511,34.64%
3,youtube,getsearch,one-way-reduced,oas,180,80,44.44%,63.0,80,511,15.66%
4,youtube,getsearch,one-way-reduced,5xx,180,4,2.22%,63.0,16,511,3.13%
5,youtube,getsearch,one-way-reduced,reg,180,156,86.67%,63.0,95,511,18.59%
6,youtube,getsearch,one-way,oas,591,262,44.33%,70.0,94,511,18.4%
7,youtube,getsearch,one-way,5xx,591,13,2.2%,70.0,26,511,5.09%
8,youtube,getsearch,one-way,reg,591,513,86.8%,70.0,119,511,23.29%


In [3]:
# 1) Columns to pin at the front
front = ['sut', 'operationId', 'strength', 'assertlevel']

# 2) Collect groups by suffix
total_cols     = [c for c in df.columns if c.endswith('_total')]
killed_cols    = [c for c in df.columns if c.endswith('_killed')]
kill_rate_cols = [c for c in df.columns if c.endswith('_kill_rate')]

# 3) Anything else (e.g. line_coverage, _passed, etc.)
others = [c for c in df.columns
          if c not in front + total_cols + killed_cols + kill_rate_cols]

# 4) Build the new column order
new_order = front + total_cols + killed_cols + kill_rate_cols + others

# 5) Reindex your DataFrame
df = df[new_order]
df

,sut,operationId,strength,assertlevel,http_total,pit_total,http_killed,pit_killed,http_kill_rate,pit_kill_rate,line_coverage
0,youtube,getsearch,two-way,oas,10522,511,4836,122,45.96%,23.87%,76.0
1,youtube,getsearch,two-way,5xx,10522,511,156,52,1.48%,10.18%,76.0
2,youtube,getsearch,two-way,reg,10522,511,9492,177,90.21%,34.64%,76.0
3,youtube,getsearch,one-way-reduced,oas,180,511,80,80,44.44%,15.66%,63.0
4,youtube,getsearch,one-way-reduced,5xx,180,511,4,16,2.22%,3.13%,63.0
5,youtube,getsearch,one-way-reduced,reg,180,511,156,95,86.67%,18.59%,63.0
6,youtube,getsearch,one-way,oas,591,511,262,94,44.33%,18.4%,70.0
7,youtube,getsearch,one-way,5xx,591,511,13,26,2.2%,5.09%,70.0
8,youtube,getsearch,one-way,reg,591,511,513,119,86.8%,23.29%,70.0


In [4]:
df.groupby(['sut', 'operationId', 'strength', 'assertlevel']).sum()

http_total  pit_total  \
sut     operationId strength        assertlevel                          
youtube getsearch   one-way         5xx                 591        511   
                                    oas                 591        511   
                                    reg                 591        511   
                    one-way-reduced 5xx                 180        511   
                                    oas                 180        511   
                                    reg                 180        511   
                    two-way         5xx               10522        511   
                                    oas               10522        511   
                                    reg               10522        511   

                                                 http_killed  pit_killed  \
sut     operationId strength        assertlevel                            
youtube getsearch   one-way         5xx                   13          26   
                                    oas                  262          94   
                                    reg                  513         119   
                    one-way-reduced 5xx                    4          16   
                                    oas                   80          80   
                                    reg                  156          95   
                    two-way         5xx                  156          52   
                                    oas                 4836         122   
                                    reg                 9492         177   

                                                http_kill_rate pit_kill_rate  \
sut     operationId strength        assertlevel                                
youtube getsearch   one-way         5xx                   2.2%         5.09%   
                                    oas                 44.33%         18.4%   
                                    reg                  86.8%        23.29%   
                    one-way-reduced 5xx                  2.22%         3.13%   
                                    oas                 44.44%        15.66%   
                                    reg                 86.67%        18.59%   
                    two-way         5xx                  1.48%        10.18%   
                                    oas                 45.96%        23.87%   
                                    reg                 90.21%        34.64%   

                                                 line_coverage  
sut     operationId strength        assertlevel                 
youtube getsearch   one-way         5xx                   70.0  
                                    oas                   70.0  
                                    reg                   70.0  
                    one-way-reduced 5xx                   63.0  
                                    oas                   63.0  
                                    reg                   63.0  
                    two-way         5xx                   76.0  
                                    oas                   76.0  
                                    reg                   76.0